In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
model = ChatOpenAI(model='gpt-4o-mini')

In [4]:
class SentimentSchema(BaseModel):
    sentiment : Literal['positive','negative'] = Field(description='Sentiment of the review')


class DiagnosisSchema(BaseModel):
    issue_type : Literal["UX","Performance","Bug","Support","Other"] = Field(description='the category of the issue mentioned in the review')
    tone : Literal["angry","calm","frustrated","disappointed"]=Field(description='The emotional tone expressed by user')
    urgency: Literal["low","medium","high"]=Field(description='How urgent or critical the issue to be ')

In [5]:
structured_model = model.with_structured_output(SentimentSchema)
diagnosis_model = model.with_structured_output(DiagnosisSchema)

In [6]:

prompt = 'what is the sentiment of the following review - The software is too good'
structured_model.invoke(prompt).sentiment

'positive'

In [7]:
class ReviewState(TypedDict):
    review : str 
    sentiment : Literal['postive','negative']
    diagnosis : dict 
    response : str 
    

In [8]:
graph = StateGraph(ReviewState)

In [9]:
def find_sentiment(state:ReviewState):
    prompt = f"for the following review find out the sentiment - {state['review']}"
    sentiment = structured_model.invoke(prompt).sentiment
    return {'sentiment':sentiment}

def check_sentiment(state: ReviewState) -> Literal['positive_response','run_diagnosis']:
    if state['sentiment'] == 'positive':
       return 'positive_response'
    else:
        return 'run_diagnosis'
def positive_response(state: ReviewState):
    prompt = f""" Write a warm thank-you message in response to this review: 
    \n\n {state['review']} \n 
    Also, Kindly ask the user to write a feedback on our Website """

    response = model.invoke(prompt)

    return {'response': response}

def run_diagnosis(state: ReviewState):
    prompt = f""" Diagnosis this negative review:\n\n {state['review'] } \n
    Return issue_type, tone and urgency.
    """
    response = diagnosis_model.invoke(prompt) # json type 

    return {'diagnosis': response.model_dump()}
def negative_response(state: ReviewState):
    diagnosis = state['diagnosis']
    prompt = f""" You are a support assistant.
    The user had a {diagnosis['issue_type']} issue, Sounded {diagnosis['tone']}, marked urgency as {diagnosis['urgency']}.
    Write an empathetic, helpful resolution message."""  
    response = model.invoke(prompt)
    return {'response': response}

In [10]:
graph.add_node('find_sentiment', find_sentiment)
graph.add_node('positive_response', positive_response)
graph.add_node('run_diagnosis', run_diagnosis)
graph.add_node('negative_response', negative_response)

graph.add_edge(START, 'find_sentiment')
graph.add_conditional_edges('find_sentiment',check_sentiment)
graph.add_edge('positive_response',END)
graph.add_edge('run_diagnosis','negative_response')
graph.add_edge('negative_response', END)

In [11]:
workflow = graph.compile()

In [12]:
initial_state = {'review':'I was genuinely impressed with how quickly and accurately this app responded to my questions. The interface is clean, easy to navigate, and the overall experience feels smooth and intuitive. The answers were relevant, well-structured, and saved me a lot of time. I have tried similar tools before, but this one stands out because of its speed and reliability. Looking forward to seeing more features in future updates. Highly recommended!'}
workflow.invoke(initial_state)

{'review': 'I was genuinely impressed with how quickly and accurately this app responded to my questions. The interface is clean, easy to navigate, and the overall experience feels smooth and intuitive. The answers were relevant, well-structured, and saved me a lot of time. I have tried similar tools before, but this one stands out because of its speed and reliability. Looking forward to seeing more features in future updates. Highly recommended!',
 'sentiment': 'positive',
 'response': AIMessage(content="Dear [User's Name],\n\nThank you so much for taking the time to share your wonderful review! We're thrilled to hear that you found our app quick, user-friendly, and reliable. It’s feedback like yours that motivates us to continuously improve and add features that you've come to expect. \n\nWe truly appreciate your support and your recommendation means the world to us. If you have a moment, we would be grateful if you could share your feedback on our website as well. Your insights can 

In [13]:
initial_state = {'review':'The app frequently gives incorrect answers and struggles to understand simple questions. I had to repeat myself multiple times, which made the experience frustrating.'}
workflow.invoke(initial_state)

{'review': 'The app frequently gives incorrect answers and struggles to understand simple questions. I had to repeat myself multiple times, which made the experience frustrating.',
 'sentiment': 'negative',
 'diagnosis': {'issue_type': 'Bug', 'tone': 'frustrated', 'urgency': 'high'},
 'response': AIMessage(content="Subject: We're Here to Help with Your Bug Issue\n\nHi [User's Name],\n\nThank you for reaching out to us, and I'm truly sorry to hear that you’re experiencing this bug. I understand how frustrating it can be, especially when it's marked as urgent. Your experience is important to us, and I’m here to help you resolve this as quickly as possible.\n\nTo assist you better, could you please provide a little more detail about the issue? Specifically, any error messages you’ve encountered or the steps you took leading up to the bug would be incredibly helpful. \n\nIn the meantime, here are a few troubleshooting steps that might resolve the issue:\n\n1. **Clear Cache:** Sometimes, cl